In [1]:
# STEP 1: Import libraries and load the raw datasets

import pandas as pd
import numpy as np
from pathlib import Path

# Project raw-data folder
data_path = Path("../../data/raw")

# Load subscription data
subscriptions = pd.read_csv(
    data_path / "subscriptions.csv"
)

# Load monthly usage data
usage = pd.read_csv(
    data_path / "usage.csv"
)

print("Subscriptions loaded successfully")
print("Usage loaded successfully")

print("\nSubscriptions shape:", subscriptions.shape)
print("Usage shape:", usage.shape)

Subscriptions loaded successfully
Usage loaded successfully

Subscriptions shape: (4200, 11)
Usage shape: (30685, 6)


In [2]:
# STEP 2: Preview datasets and inspect columns/data types

print("SUBSCRIPTIONS - First 5 Rows")
display(subscriptions.head())

print("\nUSAGE - First 5 Rows")
display(usage.head())

print("\nSUBSCRIPTIONS - Column Names")
print(subscriptions.columns.tolist())

print("\nUSAGE - Column Names")
print(usage.columns.tolist())

print("\nSUBSCRIPTIONS - Data Types")
print(subscriptions.dtypes)

print("\nUSAGE - Data Types")
print(usage.dtypes)

SUBSCRIPTIONS - First 5 Rows


,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues
1,2,2025-08-14,annual,basic,9.74,promo_offer,web,45-54,no,NaN,NaN
2,3,2025-01-24,monthly,basic,12.99,referral,ios,55+,no,NaN,NaN
3,4,2024-08-15,monthly,basic,12.99,paid_social,ios,25-34,yes,2024-08-20,not_using_enough
4,5,2025-06-24,annual,plus,14.99,paid_social,android,55+,no,NaN,NaN



USAGE - First 5 Rows


,customer_id,month,workouts_completed,minutes_active,classes_booked,support_tickets
0,1,2024-09,5,140.0,0,0
1,1,2024-10,4,116.0,0,0
2,1,2024-11,7,208.0,0,0
3,1,2024-12,5,109.0,0,0
4,1,2025-01,1,41.0,0,0



SUBSCRIPTIONS - Column Names
['customer_id', 'signup_date', 'plan_type', 'plan_tier', 'monthly_price', 'acquisition_channel', 'primary_device', 'age_group', 'churned', 'churn_date', 'cancel_reason']

USAGE - Column Names
['customer_id', 'month', 'workouts_completed', 'minutes_active', 'classes_booked', 'support_tickets']

SUBSCRIPTIONS - Data Types
customer_id              int64
signup_date                str
plan_type                  str
plan_tier                  str
monthly_price          float64
acquisition_channel        str
primary_device             str
age_group                  str
churned                    str
churn_date                 str
cancel_reason              str
dtype: object

USAGE - Data Types
customer_id             int64
month                     str
workouts_completed      int64
minutes_active        float64
classes_booked          int64
support_tickets         int64
dtype: object


In [3]:
# STEP 3: Check missing values, duplicates, and unique customers

# -----------------------------
# 1. Missing values
# -----------------------------
print("SUBSCRIPTIONS - Missing Values")
print(subscriptions.isnull().sum())

print("\nUSAGE - Missing Values")
print(usage.isnull().sum())

# -----------------------------
# 2. Duplicate rows
# -----------------------------
print("\nSUBSCRIPTIONS - Duplicate Rows")
print(subscriptions.duplicated().sum())

print("\nUSAGE - Duplicate Rows")
print(usage.duplicated().sum())

# -----------------------------
# 3. Duplicate customer IDs
# -----------------------------
print("\nSUBSCRIPTIONS - Duplicate Customer IDs")
print(subscriptions["customer_id"].duplicated().sum())

# -----------------------------
# 4. Unique customers
# -----------------------------
print("\nUnique customers in subscriptions:")
print(subscriptions["customer_id"].nunique())

print("\nUnique customers in usage:")
print(usage["customer_id"].nunique())

SUBSCRIPTIONS - Missing Values
customer_id               0
signup_date               0
plan_type                 0
plan_tier                 0
monthly_price             0
acquisition_channel       0
primary_device            0
age_group                 0
churned                   0
churn_date             2432
cancel_reason          2814
dtype: int64

USAGE - Missing Values
customer_id             0
month                   0
workouts_completed      0
minutes_active        506
classes_booked          0
support_tickets         0
dtype: int64

SUBSCRIPTIONS - Duplicate Rows
0

USAGE - Duplicate Rows
160

SUBSCRIPTIONS - Duplicate Customer IDs
0

Unique customers in subscriptions:
4200

Unique customers in usage:
4200


In [4]:
# STEP 4: Check inconsistent category labels and unusual values

# -----------------------------------
# Subscription categorical columns
# -----------------------------------

category_columns = [
    "plan_type",
    "plan_tier",
    "acquisition_channel",
    "primary_device",
    "age_group",
    "churned",
    "cancel_reason"
]

for column in category_columns:
    print(f"\n--- {column.upper()} ---")
    print(subscriptions[column].value_counts(dropna=False))


# -----------------------------------
# Check numeric ranges in subscriptions
# -----------------------------------

print("\n--- MONTHLY PRICE SUMMARY ---")
print(subscriptions["monthly_price"].describe())


# -----------------------------------
# Check numeric ranges in usage
# -----------------------------------

usage_numeric_columns = [
    "workouts_completed",
    "minutes_active",
    "classes_booked",
    "support_tickets"
]

print("\n--- USAGE NUMERIC SUMMARY ---")
print(usage[usage_numeric_columns].describe())


--- PLAN_TYPE ---
plan_type
monthly    2661
annual     1539
Name: count, dtype: int64

--- PLAN_TIER ---
plan_tier
basic      1929
plus       1462
premium     708
Basic        47
Plus         39
Premium      15
Name: count, dtype: int64

--- ACQUISITION_CHANNEL ---
acquisition_channel
organic        1298
paid_social     989
promo_offer     949
referral        488
partner         476
Name: count, dtype: int64

--- PRIMARY_DEVICE ---
primary_device
ios        1976
android    1550
web         674
Name: count, dtype: int64

--- AGE_GROUP ---
age_group
25-34    1400
35-44    1073
18-24     717
45-54     619
55+       391
Name: count, dtype: int64

--- CHURNED ---
churned
no     2432
yes    1768
Name: count, dtype: int64

--- CANCEL_REASON ---
cancel_reason
NaN                  2814
not_using_enough      570
too_expensive         322
found_alternative     175
other                 162
technical_issues      157
Name: count, dtype: int64

--- MONTHLY PRICE SUMMARY ---
count    4200.000000
mea

In [5]:
# STEP 5: Clean duplicates, category labels, and date columns

# ---------------------------------------------------
# 1. Create copies so raw data remains unchanged
# ---------------------------------------------------

subscriptions_clean = subscriptions.copy()
usage_clean = usage.copy()


# ---------------------------------------------------
# 2. Remove exact duplicate rows
# ---------------------------------------------------

print("Usage rows before duplicate removal:", len(usage_clean))

usage_clean = usage_clean.drop_duplicates()

print("Usage rows after duplicate removal:", len(usage_clean))


# ---------------------------------------------------
# 3. Standardize text/category columns
# ---------------------------------------------------

subscription_text_columns = [
    "plan_type",
    "plan_tier",
    "acquisition_channel",
    "primary_device",
    "age_group",
    "churned",
    "cancel_reason"
]

for column in subscription_text_columns:
    
    # Remove unnecessary spaces
    subscriptions_clean[column] = subscriptions_clean[column].str.strip()
    
    # Convert text to lowercase
    subscriptions_clean[column] = subscriptions_clean[column].str.lower()


# ---------------------------------------------------
# 4. Convert date columns to datetime
# ---------------------------------------------------

subscriptions_clean["signup_date"] = pd.to_datetime(
    subscriptions_clean["signup_date"],
    errors="coerce"
)

subscriptions_clean["churn_date"] = pd.to_datetime(
    subscriptions_clean["churn_date"],
    errors="coerce"
)

usage_clean["month"] = pd.to_datetime(
    usage_clean["month"],
    format="%Y-%m",
    errors="coerce"
)


# ---------------------------------------------------
# 5. Check cleaned category labels
# ---------------------------------------------------

print("\nPLAN TYPE")
print(subscriptions_clean["plan_type"].value_counts(dropna=False))

print("\nPLAN TIER")
print(subscriptions_clean["plan_tier"].value_counts(dropna=False))

print("\nCHURN STATUS")
print(subscriptions_clean["churned"].value_counts(dropna=False))


# ---------------------------------------------------
# 6. Check date types
# ---------------------------------------------------

print("\nSubscriptions date columns:")
print(
    subscriptions_clean[
        ["signup_date", "churn_date"]
    ].dtypes
)

print("\nUsage month data type:")
print(usage_clean["month"].dtype)

Usage rows before duplicate removal: 30685
Usage rows after duplicate removal: 30525

PLAN TYPE
plan_type
monthly    2661
annual     1539
Name: count, dtype: int64

PLAN TIER
plan_tier
basic      1976
plus       1501
premium     723
Name: count, dtype: int64

CHURN STATUS
churned
no     2432
yes    1768
Name: count, dtype: int64

Subscriptions date columns:
signup_date    datetime64[us]
churn_date     datetime64[us]
dtype: object

Usage month data type:
datetime64[us]


In [7]:
usage_clean = usage_clean.drop_duplicates()

In [8]:
print("Usage rows after duplicate removal:", len(usage_clean))

Usage rows after duplicate removal: 30525


In [9]:
usage_clean = usage.copy()

print("Before:", len(usage_clean))

usage_clean = usage_clean.drop_duplicates()

print("After:", len(usage_clean))

Before: 30685
After: 30525


In [10]:
# STEP 6: Handle missing values, validate data, and save cleaned datasets

# ============================================================
# 1. CHECK MISSING VALUES BEFORE CLEANING
# ============================================================

print("SUBSCRIPTIONS - Missing values before handling")
print(subscriptions_clean.isnull().sum())

print("\nUSAGE - Missing values before handling")
print(usage_clean.isnull().sum())


# ============================================================
# 2. HANDLE CANCEL REASON
# ============================================================

# Active customers cannot have a cancellation reason,
# so this missing value is expected.

subscriptions_clean.loc[
    subscriptions_clean["churned"] == "no",
    "cancel_reason"
] = "not_applicable"


# If a customer churned but did not provide a reason,
# mark it separately.

subscriptions_clean.loc[
    (subscriptions_clean["churned"] == "yes") &
    (subscriptions_clean["cancel_reason"].isna()),
    "cancel_reason"
] = "not_provided"


# ============================================================
# 3. HANDLE MISSING MINUTES_ACTIVE
# ============================================================

# Create a flag so we remember which rows originally
# had missing minutes_active values.

usage_clean["minutes_active_missing_flag"] = (
    usage_clean["minutes_active"].isna().astype(int)
)

print(
    "\nMissing minutes_active before filling:",
    usage_clean["minutes_active"].isna().sum()
)


# Calculate median minutes for each workout count
workout_median_minutes = (
    usage_clean
    .groupby("workouts_completed")["minutes_active"]
    .transform("median")
)


# Fill missing minutes using median minutes of customers
# with the same number of completed workouts
usage_clean["minutes_active"] = (
    usage_clean["minutes_active"]
    .fillna(workout_median_minutes)
)


# Fallback: if anything is still missing, use overall median
usage_clean["minutes_active"] = (
    usage_clean["minutes_active"]
    .fillna(usage_clean["minutes_active"].median())
)


print(
    "Missing minutes_active after filling:",
    usage_clean["minutes_active"].isna().sum()
)


# ============================================================
# 4. VALIDATE IMPORTANT BUSINESS RULES
# ============================================================

print("\nNegative workouts:")
print((usage_clean["workouts_completed"] < 0).sum())

print("\nNegative minutes:")
print((usage_clean["minutes_active"] < 0).sum())

print("\nNegative classes booked:")
print((usage_clean["classes_booked"] < 0).sum())

print("\nNegative support tickets:")
print((usage_clean["support_tickets"] < 0).sum())


# ============================================================
# 5. VALIDATE CHURN DATE
# ============================================================

# Churned customers should normally have a churn date

missing_churn_date = subscriptions_clean[
    (subscriptions_clean["churned"] == "yes") &
    (subscriptions_clean["churn_date"].isna())
]

print(
    "\nChurned customers without churn date:",
    len(missing_churn_date)
)


# ============================================================
# 6. FINAL DATASET SHAPES
# ============================================================

print("\nFinal subscriptions shape:")
print(subscriptions_clean.shape)

print("\nFinal usage shape:")
print(usage_clean.shape)


# ============================================================
# 7. SAVE CLEANED DATA
# ============================================================

subscriptions_clean.to_csv(
    "../../data/cleaned/subscriptions_clean.csv",
    index=False
)

usage_clean.to_csv(
    "../../data/cleaned/usage_clean.csv",
    index=False
)

print("\nCLEANED DATASETS SAVED SUCCESSFULLY")

SUBSCRIPTIONS - Missing values before handling
customer_id               0
signup_date               0
plan_type                 0
plan_tier                 0
monthly_price             0
acquisition_channel       0
primary_device            0
age_group                 0
churned                   0
churn_date             2432
cancel_reason          2814
dtype: int64

USAGE - Missing values before handling
customer_id             0
month                   0
workouts_completed      0
minutes_active        503
classes_booked          0
support_tickets         0
dtype: int64

Missing minutes_active before filling: 503
Missing minutes_active after filling: 0

Negative workouts:
0

Negative minutes:
0

Negative classes booked:
0

Negative support tickets:
0

Churned customers without churn date: 0

Final subscriptions shape:
(4200, 11)

Final usage shape:
(30525, 7)

CLEANED DATASETS SAVED SUCCESSFULLY
